# Sony PlayStation Graph — Tables Protocol

Adapts [`graph-analytics-serverless-spark.ipynb`](https://github.com/neo4j/graph-data-science-client/blob/main/examples/graph-analytics-serverless-spark.ipynb)
to a real dataset instead of the toy bike-trips example: a Sony PlayStation social graph with one
node type (`Account`) and, in this notebook, two of its relationship types (`BLOCKED`, `COMMUNICATED_WITH`).

This version uses the **tables protocol** (`create_graph` + `upload_nodes` + `upload_relationships`): `Account` nodes are uploaded explicitly, together with any properties from `accounts.parquet`, and `BLOCKED`/`COMMUNICATED_WITH` are uploaded as two separate relationship tables. This carries Account properties into the graph and guarantees every Account node exists even if it has no relationships, at the cost of more code than the triplets protocol.

This is one of two example notebooks in this repo showing the same dataset ingested two different
ways:

* `graph-analytics-sonyps-triplets.ipynb` — triplets protocol
* `graph-analytics-sonyps-tables.ipynb` — tables protocol

## Data Setup

Creates a Unity Catalog Volume and downloads the source parquet files into it, skipping any file
that has already been downloaded. `CATALOG` / `SCHEMA` / `VOLUME` are constants with sensible
defaults — edit them to point at a different location.

In [ ]:
CATALOG = "graph-on-databricks"
SCHEMA = "graph-analytics"
VOLUME = "graph-analytics-volume"

VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`")
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`.`{VOLUME}`")

import os

os.environ["VOLUME_PATH"] = VOLUME_PATH

print(f"Volume: {VOLUME_PATH}")

Download the three parquet files with a shell command. Each file is skipped if it already
exists in the Volume, so re-running this cell is safe. `communicated_with.parquet` is ~117GB, so
the download uses `curl -C -` to resume an interrupted transfer instead of restarting from zero.

In [ ]:
%sh
set -euo pipefail

BASE_URL="https://neo4j-fauth-share.s3.us-east-2.amazonaws.com/sonyplaystation"

for f in accounts.parquet blocked.parquet communicated_with.parquet; do
  dest="$VOLUME_PATH/$f"
  if [ -f "$dest" ]; then
    echo "[skip]     $f already present at $dest"
  else
    echo "[download] $f -> $dest (resumable)"
    curl -sSL -C - "$BASE_URL/$f" -o "$dest"
  fi
done

ls -la "$VOLUME_PATH"


## Reading the data into Spark

Read the three files and inspect their schemas. **Run this cell and look at the printed schemas
before trusting the column-name constants in the next cell** — these parquet files were generated
by `neo4j-admin database import`, and the exact column names depend on how that import was
configured, so the names below are a best guess, not a guarantee.

In [ ]:
accounts_df = spark.read.parquet(f"{VOLUME_PATH}/accounts.parquet")
blocked_df = spark.read.parquet(f"{VOLUME_PATH}/blocked.parquet")
communicated_with_df = spark.read.parquet(f"{VOLUME_PATH}/communicated_with.parquet")

for name, df in [("accounts", accounts_df), ("blocked", blocked_df), ("communicated_with", communicated_with_df)]:
    print(f"--- {name} ---")
    df.printSchema()
    df.show(5)

Adjust these to match the column names printed above.

In [ ]:
# Node id column in accounts.parquet
ACCOUNT_ID_COL = "accountId"

# Source/target account id columns in each relationship file
BLOCKED_SOURCE_COL = "sourceAccountId"
BLOCKED_TARGET_COL = "targetAccountId"

COMMUNICATED_WITH_SOURCE_COL = "sourceAccountId"
COMMUNICATED_WITH_TARGET_COL = "targetAccountId"

## Prerequisites

We also need to have the `graphdatascience` Python library installed, version `2.0a1` or later, as well as `pyspark`.

In [ ]:
%pip install "graphdatascience>=2.0a1" python-dotenv "pyspark[sql]"

In [ ]:
from dotenv import load_dotenv

# This allows to load required secrets from `.env` file in local directory
# This can include Aura API Credentials and Database Credentials.
# If file does not exist this is a noop.
load_dotenv("sessions.env")

### Spark session

Databricks already provides a `spark` session, so we only need to enable Arrow-based columnar
transfers on it (no need to build a new local `SparkSession` as the upstream notebook does).

In [ ]:
# Enable Arrow-based columnar data transfers
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

## Aura API credentials

The entry point for managing GDS Sessions is the `GdsSessions` object, which requires creating [Aura API credentials](https://neo4j.com/docs/aura/api/authentication).

In [ ]:
from graphdatascience.session import AuraAPICredentials, GdsSessions

# you can also use AuraAPICredentials.from_env() to load credentials from environment variables
api_credentials = AuraAPICredentials(
    client_id=os.environ["CLIENT_ID"],
    client_secret=os.environ["CLIENT_SECRET"],
    # If your account is a member of several projects, you must also specify the project ID to use
    project_id=os.environ.get("PROJECT_ID", None),
)

sessions = GdsSessions(api_credentials=api_credentials)

## Sizing and creating the session

`communicated_with.parquet` alone is ~117GB raw, far bigger than the `SessionMemory.m_2GB` tier
used for the toy bike-trips example. Rather than guess a tier, we ask Aura to recommend one via
`sessions.estimate()`, based on the actual row counts of our node and relationship data.

In [ ]:
node_count = accounts_df.count()
blocked_count = blocked_df.count()
communicated_with_count = communicated_with_df.count()
relationship_count = blocked_count + communicated_with_count

print(f"accounts: {node_count}, blocked: {blocked_count}, communicated_with: {communicated_with_count}")

recommended_memory = sessions.estimate(
    node_count=node_count,
    relationship_count=relationship_count,
)
print(f"Recommended session memory: {recommended_memory}")

In [ ]:
from datetime import timedelta

from graphdatascience.session import CloudLocation

# Create a GDS session!
gds = sessions.get_or_create(
    session_name="sonyps_accounts",
    memory=recommended_memory,
    ttl=timedelta(minutes=30),
    cloud_location=CloudLocation("gcp", "europe-west1"),
)

In [ ]:
# Verify the connectivity. Hints towards TLS or firewall issues if this fails directly after get_or_create
gds.verify_connectivity()

## Projecting the graph (tables protocol)

We first need to get access to the `GdsArrowClient`. This client allows us to directly communicate
with the Arrow Flight server provided by the session.

The tables protocol loads nodes and relationships as two explicit phases within one job:

1. Send `v2/graph.project.fromTables` to get a `job_id`.
2. Stream `Account` node batches (`nodeId` + any properties) to the Arrow server, then call
   `node_load_done`.
3. Wait for the job to reach the `RELATIONSHIP_LOADING` status.
4. Stream `BLOCKED` and `COMMUNICATED_WITH` relationship batches — each Spark DataFrame is
   homogeneous, so we run one `mapInArrow` upload pass per relationship type, both writing into
   the same `job_id` — then call `relationship_load_done`.
5. Wait for the import process to reach the `Done` state.

In [ ]:
import time

import pandas as pd
import pyarrow
from pyspark.sql import functions as F

graph_name = "sonyps_tables"

arrow_client = gds.arrow_client()

# 1. Start the import process
job_id = arrow_client.create_graph(graph_name)


def upload_node_batch(iterator):
    for batch in iterator:
        arrow_client.upload_nodes(job_id, [batch])
        yield pyarrow.RecordBatch.from_pandas(pd.DataFrame({"batch_rows_imported": [len(batch)]}))


# 2. Build the Account node table: nodeId + labels + any remaining properties from accounts.parquet
property_cols = [c for c in accounts_df.columns if c != ACCOUNT_ID_COL]

account_nodes = accounts_df.select(
    F.col(ACCOUNT_ID_COL).alias("nodeId"),
    F.lit("Account").alias("labels"),
    *property_cols,
)

uploaded_node_batches = account_nodes.mapInArrow(upload_node_batch, "batch_rows_imported long")
uploaded_node_batches.agg(F.sum("batch_rows_imported").alias("rows_imported")).show()

arrow_client.node_load_done(job_id)

# 3. Wait until the job is ready to accept relationships
while arrow_client.job_status(job_id).status != "RELATIONSHIP_LOADING":
    time.sleep(1)

In [ ]:
def upload_rel_batch(iterator):
    for batch in iterator:
        arrow_client.upload_relationships(job_id, [batch])
        yield pyarrow.RecordBatch.from_pandas(pd.DataFrame({"batch_rows_imported": [len(batch)]}))


# 4. Build and upload each relationship type as its own homogeneous DataFrame
blocked_rels = blocked_df.select(
    F.col(BLOCKED_SOURCE_COL).alias("sourceNodeId"),
    F.col(BLOCKED_TARGET_COL).alias("targetNodeId"),
).withColumn("relationshipType", F.lit("BLOCKED"))

communicated_with_rels = communicated_with_df.select(
    F.col(COMMUNICATED_WITH_SOURCE_COL).alias("sourceNodeId"),
    F.col(COMMUNICATED_WITH_TARGET_COL).alias("targetNodeId"),
).withColumn("relationshipType", F.lit("COMMUNICATED_WITH"))

for rel_df in [blocked_rels, communicated_with_rels]:
    uploaded_rel_batches = rel_df.mapInArrow(upload_rel_batch, "batch_rows_imported long")
    uploaded_rel_batches.agg(F.sum("batch_rows_imported").alias("rows_imported")).show()

arrow_client.relationship_load_done(job_id)

# 5. Wait for the import to finish
while arrow_client.job_status(job_id).status != "Done":
    time.sleep(1)

G = gds.graph.get(graph_name)
G

## Running Algorithms

We can run algorithms on the constructed graph using the standard GDS Python Client API. See the other tutorials for more examples.

In [ ]:
print("Running PageRank ...")
pr_result = gds.page_rank.mutate(G, mutate_property="pagerank")

## Sending the computation result back to Spark

Once the computation is done, we might want to further use the result in Spark.
We can do this in a similar way to the projection, by streaming batches of data into each of the Spark workers.
Retrieving the data is a bit more complicated since we need some input DataFrame in order to trigger computations on the Spark workers.
We use a data range equal to the size of workers we have in our cluster as our driving table.
On the workers we will disregard the input and instead stream the computation data from the GDS Session.

In [ ]:
import pyarrow

# 1. Start the node property export on the GDS session
export_job_id = arrow_client.get_node_properties(G.name(), ["pagerank"])


# Define a function that receives data from the GDS Session and turns it into data batches
def retrieve_data(ignored):
    stream_data = arrow_client.stream_job(export_job_id)
    batches = pyarrow.Table.from_pandas(stream_data).to_batches(1000)
    for b in batches:
        yield b


# Create DataFrame with a single column and one row per worker
input_partitions = spark.range(spark.sparkContext.defaultParallelism).toDF("batch_id")
# 2. Stream the data from the GDS Session into the Spark workers
received_batches = input_partitions.mapInArrow(retrieve_data, "nodeId long, pagerank double")
# Optional: Repartition the data to make sure it is distributed equally
result = received_batches.repartition(numPartitions=spark.sparkContext.defaultParallelism)

result.toPandas()

## Cleanup

Now that we have finished our analysis, we can delete the GDS session.

Deleting the GDS session will release all resources associated with it, and stop incurring costs.

In [ ]:
gds.delete()